In [ ]:
import wandb
import pandas as pd
from tqdm import tqdm
import numpy as np

def get_dataframe_from_records(records):
    """Convert records to a DataFrame."""
    
    # Create DataFrame
    df = pd.DataFrame(records)

    # Group by group and calculate mean and std
    agg_df = df.groupby("group").agg(['mean', 'std'])

    # format to mean /pm std
    # Format as 'mean ± std' rounded to 3 decimals
    formatted_df = agg_df.copy()
    for metric in df.columns[1:]:  # Skip 'group' column
        mean_col = (metric, 'mean')
        std_col = (metric, 'std')
        formatted_col_name = f"{metric} (mean ± std)"

        formatted_df[formatted_col_name] = agg_df.apply(
            lambda row: f"{row[mean_col]:.3f} ± {row[std_col]:.3f}", axis=1
        )
    # Optional: flatten the column names
    # agg_df.columns = ['_'.join(col) for col in agg_df.columns]
    # remove original colums
    formatted_df = formatted_df.drop(columns=[(col, 'mean') for col in df.columns[1:]] + [(col, 'std') for col in df.columns[1:]])
    return formatted_df

def get_records(project, metrics=[]):
    # Authenticate (you must be logged in via wandb.login())
    api = wandb.Api()

    # Replace with your entity/project
    entity = "ecg_ml"

    # Fetch all runs
    runs = api.runs(f"{entity}/{project}")

    # Collect relevant info
    records = []
    for run in tqdm(runs):
        if run.state != "finished":
            continue

        # add only if group name starts with best_
        if not run.group or not run.group.startswith("best_"):
            continue

        # Get the group name
        group = run.group or "ungrouped"

        record = {
            "group": group
        }

        # Extract your test metrics — replace with your actual metric names
        for metric in metrics:
            if metric not in run.summary:
                print(f"Warning: Metric '{metric}' not found in run {run.id}.")
                continue
            record[metric] = run.summary.get(metric)

        records.append(record)
    return records

def style_mean_column(col):
    # Extract float values from the strings
    mean_vals = col.str.extract(r"^([\d.]+)")[0].astype(float)
    
    # Get indices of highest and second highest
    top2_idx = mean_vals.nlargest(2).index

    # Build styles
    styles = []
    for i in col.index:
        if i == top2_idx[0]:
            styles.append("font-weight: bold")
        elif i == top2_idx[1]:
            styles.append("text-decoration: underline")
        else:
            styles.append("")
    return styles


# PTB-XL metrics

In [2]:
metrics = ["test_acc", "test_f1", "test_auroc", "test_auprc"]
records = get_records("train-ptbxl-diagnosis_superclass-multilabel", metrics=metrics)
formatted_df = get_dataframe_from_records(records)

100%|██████████| 457/457 [01:07<00:00,  6.73it/s]


## Finetuning

In [6]:
agg_df_full_ft = formatted_df[formatted_df.index.str.endswith("_ft")]
styled_df = agg_df_full_ft.style.apply(style_mean_column)
styled_df

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std),test_auprc (mean ± std)
,,,,
group,,,,
best_ecgfm_ft,0.891 ± 0.001,0.734 ± 0.002,0.930 ± 0.000,0.822 ± 0.001
best_jepa_ft,0.884 ± 0.001,0.733 ± 0.003,0.923 ± 0.000,0.809 ± 0.000
best_stmem_ft,0.892 ± 0.001,0.739 ± 0.005,0.930 ± 0.001,0.822 ± 0.002
best_trans_ft,0.880 ± 0.002,0.710 ± 0.017,0.925 ± 0.001,0.808 ± 0.001
best_xlstm_ft,0.885 ± 0.001,0.720 ± 0.013,0.928 ± 0.001,0.817 ± 0.003


## Linear probing

In [8]:
agg_df_full_lp = formatted_df[formatted_df.index.str.endswith("_lp")]
styled_df_lp = agg_df_full_lp.style.apply(style_mean_column)
styled_df_lp

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std),test_auprc (mean ± std)
,,,,
group,,,,
best_ecgfm_lp,0.883 ± 0.001,0.702 ± 0.002,0.917 ± 0.000,0.799 ± 0.000
best_jepa_lp,0.864 ± 0.001,0.712 ± 0.002,0.902 ± 0.000,0.769 ± 0.000
best_stmem_lp,0.847 ± 0.001,0.564 ± 0.002,0.867 ± 0.001,0.693 ± 0.001
best_trans_lp,0.875 ± 0.001,0.703 ± 0.002,0.914 ± 0.000,0.782 ± 0.001
best_xlstm_lp,0.876 ± 0.001,0.690 ± 0.001,0.915 ± 0.000,0.788 ± 0.001


## Linear probing 10%

In [9]:
agg_df_10 = formatted_df[formatted_df.index.str.endswith("_10")]
styled_df_10 = agg_df_10.style.apply(style_mean_column)
styled_df_10

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std),test_auprc (mean ± std)
,,,,
group,,,,
best_ecgfm_lp_10,0.868 ± 0.001,0.616 ± 0.006,0.891 ± 0.001,0.745 ± 0.004
best_jepa_lp_10,0.789 ± 0.003,0.638 ± 0.003,0.878 ± 0.002,0.711 ± 0.011
best_stmem_lp_10,0.808 ± 0.001,0.314 ± 0.015,0.816 ± 0.002,0.565 ± 0.007
best_trans_lp_10,0.862 ± 0.001,0.663 ± 0.004,0.893 ± 0.001,0.741 ± 0.002
best_xlstm_lp_10,0.859 ± 0.002,0.613 ± 0.002,0.884 ± 0.002,0.718 ± 0.010


## Linear probing 1%

In [10]:
agg_df_1 = formatted_df[formatted_df.index.str.endswith("_1")]
styled_df_1 = agg_df_1.style.apply(style_mean_column)
styled_df_1

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std),test_auprc (mean ± std)
,,,,
group,,,,
best_ecgfm_lp_1,0.855 ± nan,0.637 ± nan,0.874 ± nan,0.702 ± nan
best_jepa_lp_1,0.722 ± 0.012,0.571 ± 0.010,0.825 ± 0.008,0.615 ± 0.018
best_stmem_lp_1,0.799 ± 0.003,0.343 ± 0.014,0.769 ± 0.004,0.506 ± 0.009
best_trans_lp_1,0.749 ± 0.015,0.595 ± 0.009,0.841 ± 0.006,0.633 ± 0.010
best_xlstm_lp_1,0.827 ± 0.007,0.526 ± 0.042,0.836 ± 0.008,0.621 ± 0.009


# CPSC 2018

In [12]:
records_cpsc = get_records("train-cpsc2018-multilabel", metrics=["test_acc", "test_f1", "test_auroc", "test_auprc"])
formatted_df_cpsc = get_dataframe_from_records(records_cpsc)

100%|██████████| 302/302 [00:42<00:00,  7.14it/s]


## Finetuning

In [13]:
formatted_df_cpsc_ft = formatted_df_cpsc[formatted_df_cpsc.index.str.endswith("_ft")]
styled_df_ft = formatted_df_cpsc_ft.style.apply(style_mean_column)
styled_df_ft

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std),test_auprc (mean ± std)
,,,,
group,,,,
best_ecgfm_ft,0.964 ± 0.000,0.794 ± 0.005,0.969 ± 0.001,0.854 ± 0.002
best_jepa_ft,0.942 ± 0.003,0.749 ± 0.009,0.965 ± 0.001,0.842 ± 0.003
best_stmem_ft,0.963 ± 0.001,0.789 ± 0.012,0.958 ± 0.003,0.835 ± 0.011
best_trans_ft,0.961 ± 0.001,0.796 ± 0.008,0.965 ± 0.003,0.853 ± 0.006
best_xlstm_ft,0.968 ± 0.001,0.822 ± 0.008,0.981 ± 0.001,0.888 ± 0.003


## Linear probing

In [14]:
formatted_df_cpsc_lp = formatted_df_cpsc[formatted_df_cpsc.index.str.endswith("_lp")]
styled_df_lp = formatted_df_cpsc_lp.style.apply(style_mean_column)
styled_df_lp


,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std),test_auprc (mean ± std)
,,,,
group,,,,
best_ecgfm_lp,0.958 ± 0.000,0.750 ± 0.007,0.962 ± 0.000,0.830 ± 0.001
best_jepa_lp,0.889 ± 0.002,0.613 ± 0.003,0.961 ± 0.000,0.830 ± 0.001
best_stmem_lp,0.935 ± 0.000,0.455 ± 0.005,0.922 ± 0.001,0.722 ± 0.004
best_trans_lp,0.953 ± 0.001,0.733 ± 0.008,0.951 ± 0.001,0.787 ± 0.008
best_xlstm_lp,0.961 ± 0.002,0.781 ± 0.011,0.968 ± 0.001,0.861 ± 0.004


## Linear probing 10%


In [15]:
formatted_df_cpsc_10 = formatted_df_cpsc[formatted_df_cpsc.index.str.endswith("_10")]
styled_df_cpsc_10 = formatted_df_cpsc_10.style.apply(style_mean_column)
styled_df_cpsc_10

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std),test_auprc (mean ± std)
,,,,
group,,,,
best_ecgfm_lp_10,0.913 ± 0.002,0.182 ± 0.011,0.919 ± 0.003,0.672 ± 0.012
best_jepa_lp_10,0.748 ± 0.004,0.451 ± 0.003,0.934 ± 0.007,0.763 ± 0.013
best_stmem_lp_10,0.885 ± 0.003,0.027 ± 0.016,0.798 ± 0.010,0.448 ± 0.005
best_trans_lp_10,0.789 ± 0.013,0.461 ± 0.011,0.898 ± 0.006,0.653 ± 0.012
best_xlstm_lp_10,0.930 ± 0.002,0.406 ± 0.025,0.913 ± 0.007,0.689 ± 0.027


## Linear probing 1%

In [16]:
formatted_df_cpsc_1 = formatted_df_cpsc[formatted_df_cpsc.index.str.endswith("_1")]
styled_df_cpsc_1 = formatted_df_cpsc_1.style.apply(style_mean_column)
styled_df_cpsc_1

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std),test_auprc (mean ± std)
,,,,
group,,,,
best_ecgfm_lp_1,0.933 ± 0.005,0.481 ± 0.073,0.920 ± 0.007,0.678 ± 0.012
best_jepa_lp_1,0.674 ± 0.007,0.384 ± 0.007,0.878 ± 0.010,0.598 ± 0.020
best_stmem_lp_1,0.908 ± 0.006,0.250 ± 0.043,0.827 ± 0.012,0.479 ± 0.024
best_trans_lp_1,0.714 ± 0.021,0.389 ± 0.017,0.827 ± 0.013,0.515 ± 0.011
best_xlstm_lp_1,0.915 ± 0.010,0.401 ± 0.086,0.873 ± 0.015,0.591 ± 0.029


# MIT-BIH classification

In [23]:
records_mit_cls = get_records("train-mitbih-5", metrics=["test_acc", "test_sensitivity/mean", "test_ppv/mean", "test_specificity/mean", "test_f1/mean"])
formatted_df_mit_cls = get_dataframe_from_records(records_mit_cls)

100%|██████████| 274/274 [00:38<00:00,  7.13it/s]


## Finetuning

In [24]:
formatted_mit_ft = formatted_df_mit_cls[formatted_df_mit_cls.index.str.endswith("_ft")]
styled_df_mit_ft = formatted_mit_ft.style.apply(style_mean_column)
styled_df_mit_ft

,test_acc (mean ± std),test_sensitivity/mean (mean ± std),test_ppv/mean (mean ± std),test_specificity/mean (mean ± std),test_f1/mean (mean ± std)
,,,,,
group,,,,,
best_ecgfm_ft,0.863 ± 0.008,0.211 ± 0.002,0.207 ± 0.002,0.809 ± 0.002,0.207 ± 0.002
best_jepa_ft,0.954 ± 0.001,0.555 ± 0.007,0.626 ± 0.007,0.950 ± 0.001,0.581 ± 0.007
best_stmem_ft,0.967 ± 0.002,0.624 ± 0.007,0.671 ± 0.013,0.964 ± 0.003,0.644 ± 0.007
best_trans_ft,0.896 ± 0.008,0.475 ± 0.002,0.457 ± 0.005,0.905 ± 0.001,0.425 ± 0.006
best_xlstm_ft,0.976 ± 0.007,0.651 ± 0.035,0.721 ± 0.019,0.973 ± 0.010,0.677 ± 0.025


## Linear probing

In [25]:
formatted_mit_lp = formatted_df_mit_cls[formatted_df_mit_cls.index.str.endswith("_lp")]
styled_df_mit_lp = formatted_mit_lp.style.apply(style_mean_column)
styled_df_mit_lp


,test_acc (mean ± std),test_sensitivity/mean (mean ± std),test_ppv/mean (mean ± std),test_specificity/mean (mean ± std),test_f1/mean (mean ± std)
,,,,,
group,,,,,
best_ecgfm_lp,0.869 ± 0.012,0.210 ± 0.003,0.210 ± 0.004,0.807 ± 0.003,0.206 ± 0.004
best_jepa_lp,0.946 ± 0.000,0.470 ± 0.003,0.574 ± 0.003,0.933 ± 0.000,0.496 ± 0.003
best_stmem_lp,0.936 ± 0.003,0.414 ± 0.012,0.503 ± 0.032,0.920 ± 0.002,0.424 ± 0.019
best_trans_lp,0.931 ± 0.000,0.381 ± 0.004,0.549 ± 0.012,0.924 ± 0.004,0.395 ± 0.005
best_xlstm_lp,0.972 ± 0.002,0.686 ± 0.021,0.669 ± 0.014,0.976 ± 0.003,0.674 ± 0.013


# MIT-BIH r-peaks

In [27]:
records_mit_r_peaks = get_records("train-mitbih-r_peaks", metrics=["test_f1_150", "test_f1_20", "test_avg_total_distance"])
formatted_df_mit_r_peaks = get_dataframe_from_records(records_mit_r_peaks)

100%|██████████| 128/128 [00:13<00:00,  9.73it/s]


## Fine-tuning

In [28]:
formatted_df_mit_r_peaks_ft = formatted_df_mit_r_peaks[formatted_df_mit_r_peaks.index.str.endswith("_ft")]
styled_df_mit_r_peaks_ft = formatted_df_mit_r_peaks_ft.style.apply(style_mean_column)
styled_df_mit_r_peaks_ft

,test_f1_150 (mean ± std),test_f1_20 (mean ± std),test_avg_total_distance (mean ± std)
,,,
group,,,
best_ecgfm_ft,0.081 ± 0.006,0.011 ± 0.001,3794.272 ± 789.848
best_jepa_ft,0.996 ± 0.000,0.909 ± 0.004,4.195 ± 0.059
best_stmem_ft,0.994 ± 0.001,0.937 ± 0.001,5.148 ± 0.403
best_trans_ft,0.983 ± 0.001,0.900 ± 0.001,7.587 ± 0.159
best_xlstm_ft,0.995 ± 0.000,0.921 ± 0.001,5.208 ± 0.098


## Linear probing

In [30]:
formatted_df_mit_r_peaks_lp = formatted_df_mit_r_peaks[formatted_df_mit_r_peaks.index.str.endswith("_lp")]
styled_df_mit_r_peaks_lp = formatted_df_mit_r_peaks_lp.style.apply(style_mean_column)
styled_df_mit_r_peaks_lp

,test_f1_150 (mean ± std),test_f1_20 (mean ± std),test_avg_total_distance (mean ± std)
,,,
group,,,
best_ecgfm_lp,0.073 ± 0.005,0.010 ± 0.001,4564.209 ± 630.534
best_jepa_lp,0.976 ± 0.000,0.444 ± 0.001,6.858 ± 0.016
best_stmem_lp,0.970 ± 0.001,0.564 ± 0.002,7.430 ± 0.139
best_trans_lp,0.864 ± 0.001,0.476 ± 0.000,32.177 ± 0.086
best_xlstm_lp,0.945 ± 0.005,0.590 ± 0.011,11.487 ± 0.283


# Exercise high intensity r peaks

In [31]:
records_exe_r = get_records("train-exercise-r_peak", metrics=["test_f1_150", "test_f1_20", "test_avg_total_distance"])
formatted_df_exe_r = get_dataframe_from_records(records_exe_r)

100%|██████████| 120/120 [00:12<00:00,  9.99it/s]


# Finetuning

In [32]:
formatted_exe_r_ft = formatted_df_exe_r[formatted_df_exe_r.index.str.endswith("_ft")]
styled_df_exe_r_ft = formatted_exe_r_ft.style.apply(style_mean_column)
styled_df_exe_r_ft

,test_f1_150 (mean ± std),test_f1_20 (mean ± std),test_avg_total_distance (mean ± std)
,,,
group,,,
best_ecgfm_ft,0.426 ± 0.006,0.082 ± 0.005,53.058 ± 1.501
best_jepa_ft,0.975 ± 0.002,0.857 ± 0.008,3.716 ± 0.161
best_trans_ft,0.850 ± 0.006,0.631 ± 0.011,12.475 ± 0.484
best_xlstm_ft,0.996 ± 0.001,0.968 ± 0.002,3.983 ± 0.231


# Linear probing

In [33]:
formatted_exe_r_lp = formatted_df_exe_r[formatted_df_exe_r.index.str.endswith("_lp")]
styled_df_exe_r_lp = formatted_exe_r_lp.style.apply(style_mean_column)
styled_df_exe_r_lp

,test_f1_150 (mean ± std),test_f1_20 (mean ± std),test_avg_total_distance (mean ± std)
,,,
group,,,
best_ecgfm_lp,0.420 ± 0.016,0.082 ± 0.002,52.658 ± 3.061
best_jepa_lp,0.924 ± 0.004,0.525 ± 0.007,13.473 ± 0.940
best_trans_lp,0.605 ± 0.006,0.302 ± 0.002,25.289 ± 0.381
best_xlstm_lp,0.948 ± 0.004,0.891 ± 0.004,17.983 ± 1.049


# Sleep apnea

In [34]:
records_sleep_apnea = get_records("train-sleep-apnea", metrics=["test_acc", "test_f1", "test_auc"])
formatted_df_sleep_apnea = get_dataframe_from_records(records_sleep_apnea)

100%|██████████| 188/188 [00:23<00:00,  8.05it/s]


# Fine-tuning

In [35]:
formatted_sleep_ft = formatted_df_sleep_apnea[formatted_df_sleep_apnea.index.str.endswith("_ft")]
styled_df_sleep_ft = formatted_sleep_ft.style.apply(style_mean_column)
styled_df_sleep_ft

,test_acc (mean ± std),test_f1 (mean ± std),test_auc (mean ± std)
,,,
group,,,
best_ecgfm_ft,0.575 ± 0.018,0.566 ± 0.017,0.580 ± 0.021
best_jepa_ft,0.628 ± nan,0.600 ± nan,0.601 ± nan
best_stmem_ft,0.604 ± 0.029,0.600 ± 0.033,0.727 ± 0.035
best_trans_ft,0.563 ± 0.033,0.558 ± 0.030,0.604 ± 0.058
best_xlstm_ft,0.855 ± nan,0.808 ± nan,0.871 ± nan


# Linear probing

In [36]:
formatted_sleep_lp = formatted_df_sleep_apnea[formatted_df_sleep_apnea.index.str.endswith("_lp")]
styled_df_sleep_lp = formatted_sleep_lp.style.apply(style_mean_column)
styled_df_sleep_lp

,test_acc (mean ± std),test_f1 (mean ± std),test_auc (mean ± std)
,,,
group,,,
best_ecgfm_lp,0.555 ± 0.060,0.548 ± 0.067,0.579 ± 0.082
best_jepa_lp,0.495 ± nan,0.489 ± nan,0.543 ± nan
best_stmem_lp,0.621 ± 0.018,0.616 ± 0.020,0.642 ± 0.030
best_trans_lp,0.683 ± 0.025,0.656 ± 0.028,0.649 ± 0.029
best_xlstm_lp,0.722 ± nan,0.597 ± nan,0.776 ± nan


# PPG

In [37]:
records_ppg = get_records("train-deepbeat", metrics=["test_acc", "test_f1", "test_auroc"])
formatted_df_ppg = get_dataframe_from_records(records_ppg)

100%|██████████| 142/142 [00:15<00:00,  9.12it/s]


## Fine-tuning

In [38]:
formatted_ppg_ft = formatted_df_ppg[formatted_df_ppg.index.str.endswith("_ft")]
styled_df_ppg_ft = formatted_ppg_ft.style.apply(style_mean_column)
styled_df_ppg_ft

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std)
,,,
group,,,


## Linear probing

In [39]:
formatted_df_ppg_lp = formatted_df_ppg[formatted_df_ppg.index.str.endswith("_lp")]
styled_df_ppg_lp = formatted_df_ppg_lp.style.apply(style_mean_column)
styled_df_ppg_lp

,test_acc (mean ± std),test_f1 (mean ± std),test_auroc (mean ± std)
,,,
group,,,
best_jepa_lp,0.520 ± 0.009,0.447 ± 0.010,0.477 ± 0.007
